In [ ]:
# Change this to your preferred framework (e.g., 'cuda', 'pytorch', 'triton', 'jax', 'mojo')
EVAL_LANG = 'cuda'

SAVE_GPU = True


<p>
  Solve the logistic regression problem on a GPU. Given a feature matrix $X$ of size $n\_samples \times n\_features$ and a binary target vector $y$ of size $n\_samples$ (containing only 0s and 1s), compute the coefficient vector $\beta$ that maximizes the log-likelihood:
  $$ \max_{\beta} \sum_{i=1}^{n} \left[ y_i \log(p_i) + (1-y_i) \log(1-p_i) \right] $$

  where $p_i = \sigma(X_i^T \beta)$ and $\sigma(z) = \frac{1}{1 + e^{-z}}$ is the sigmoid function.
</p>

<h2>Implementation Requirements</h2>
<ul>
  <li>External libraries are not permitted</li>
  <li>The <code>solve</code> function signature must remain unchanged</li>
  <li>The final coefficients must be stored in the <code>beta</code> vector</li>
  <li>The target vector <code>y</code> contains only binary values (0 and 1)</li>
</ul>

<h2>Example:</h2>
<p>
Input:<br>
$X$ (samples × features):
$$
\begin{bmatrix}
2.0 & 1.0 \\
1.0 & 2.0 \\
3.0 & 3.0 \\
1.5 & 2.5 \\
-1.0 & -2.0 \\
-2.0 & -1.0 \\
-1.5 & -2.5 \\
-3.0 & -3.0
\end{bmatrix}
$$
$y$:
$$
\begin{bmatrix}
1 \\
1 \\
1 \\
0 \\
0 \\
0 \\
1 \\
0
\end{bmatrix}
$$
Output:<br>
$\beta$:
$$
\begin{bmatrix}
2.26 \\
-1.29
\end{bmatrix}
$$
</p>

<h2>Constraints</h2>
<ul>
  <li>1 ≤ <code>n_samples</code> ≤ 100,000</li>
  <li>1 ≤ <code>n_features</code> ≤ 1,000</li>
  <li><code>n_samples</code> ≥ <code>n_features</code></li>
  <li>-10.0 ≤ values in <code>X</code> ≤ 10.0</li>
  <li><code>y</code> contains only binary values: 0 or 1</li>
  <li>Solutions are tested with absolute tolerance of 1e-2 and relative tolerance of 1e-2</li>

  <li>Performance is measured with <code>n_features</code> = 8, <code>n_samples</code> = 16</li>
</ul>


# CUDA

In [ ]:
%%writefile solution.cu
#include <cuda_runtime.h>

// X, y, beta are device pointers
extern "C" void solve(const float* X, const float* y, float* beta, int n_samples, int n_features) {}


# CUTE

In [ ]:
%%writefile solution_cute.py
import cutlass
import cutlass.cute as cute


# X, y, beta are tensors on the GPU
@cute.jit
def solve(
    X: cute.Tensor, y: cute.Tensor, beta: cute.Tensor, n_samples: cute.Int32, n_features: cute.Int32
):
    pass


# JAX

In [ ]:
%%writefile solution_jax.py
import jax
import jax.numpy as jnp


# X, y are tensors on the GPU
@jax.jit
def solve(X: jax.Array, y: jax.Array, n_samples: int, n_features: int) -> jax.Array:
    # return output tensor directly
    pass


# MOJO

In [ ]:
%%writefile solution.mojo
from std.gpu.host import DeviceContext
from std.gpu import block_dim, block_idx, thread_idx
from std.memory import UnsafePointer
from std.math import ceildiv


# X, y, beta are device pointers (i.e. pointers to memory on the GPU)
@export
def solve(
    X: UnsafePointer[Float32, MutExternalOrigin],
    y: UnsafePointer[Float32, MutExternalOrigin],
    beta: UnsafePointer[Float32, MutExternalOrigin],
    n_samples: Int32,
    n_features: Int32,
) raises:
    pass


# Torch

In [ ]:
%%writefile solution_pytorch.py
import torch


# X, y, beta are tensors on the GPU
def solve(X: torch.Tensor, y: torch.Tensor, beta: torch.Tensor, n_samples: int, n_features: int):
    pass


# Triton

In [ ]:
%%writefile solution_triton.py
import torch
import triton
import triton.language as tl


# X, y, beta are tensors on the GPU
def solve(X: torch.Tensor, y: torch.Tensor, beta: torch.Tensor, n_samples: int, n_features: int):
    pass


# Evaluate Setup

In [ ]:
# Download required files from GitHub
!mkdir -p core
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/core/challenge_base.py -O core/challenge_base.py
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/core/evaluator.py -O core/evaluator.py
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/medium/34_logistic_regression/challenge.py -O challenge.py

from challenge import Challenge
from core.evaluator import Evaluate

ch = Challenge()


# Evaluation code

In [ ]:
# Run the evaluator based on configuration
if EVAL_LANG == 'cuda':
    Evaluate.eval_cuda(ch)
elif EVAL_LANG in ['pytorch', 'triton', 'jax', 'cute']:
    Evaluate.eval_python(ch, EVAL_LANG)
elif EVAL_LANG == 'mojo':
    Evaluate.eval_mojo(ch)
else:
    print(f"Unknown language {EVAL_LANG}")

# Disconnect runtime to save Colab resources
if SAVE_GPU:
    from google.colab import runtime
    runtime.unassign()
